# UEFA Euro 2024 Final — Visualizations
**Spain 2–1 England** | Olympiastadion, Berlin | 14 July 2024

Loads processed events from `cache/` and saves all figures to `figures/`.

Goals:
- 46' Nico Williams (Spain)
- 72' Cole Palmer (England)
- 85' Mikel Oyarzabal (Spain)

**360 / freeze-frame data:** Not available for UEFA Euro 2024 in StatsBomb open data. All spatial figures use event x/y coordinates only.

## 0. Setup & Data Load

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as mpe
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde
from scipy.ndimage import gaussian_filter
from mplsoccer import Pitch, VerticalPitch
warnings.filterwarnings('ignore')

BASE  = r'D:\Masaüstü\Projects\MatchAnalysis\Spain-England2024'
CACHE = os.path.join(BASE, 'cache')
FIGS  = os.path.join(BASE, 'figures')
os.makedirs(FIGS, exist_ok=True)

PITCH_COLOR = '#1a1a2e'
LINE_COLOR  = 'white'
ESP_COLOR   = '#c60b1e'
ENG_COLOR   = '#003090'
DPI = 150

def save(name):
    path = os.path.join(FIGS, name)
    plt.savefig(path, dpi=DPI, bbox_inches='tight', facecolor=plt.gcf().get_facecolor())
    plt.close('all')
    size = os.path.getsize(path) // 1024
    print(f'  saved {name}  ({size} KB)')

# Load
events = pd.read_pickle(os.path.join(CACHE, 'events_processed.pkl'))
with open(os.path.join(CACHE, 'meta.json'), encoding='utf-8') as f:
    meta = json.load(f)

ESP = meta['ESP']   # 'Spain'
ENG = meta['ENG']   # 'England'

shots      = events[events['type'] == 'Shot'].copy()
passes     = events[events['type'] == 'Pass'].copy()
carries    = events[events['type'] == 'Carry'].copy()
presses    = events[events['type'] == 'Pressure'].copy()
blocks     = events[events['type'] == 'Block'].copy()
intercepts = events[events['type'] == 'Interception'].copy()
touches    = pd.concat([passes[['team','player','x','y']],
                        carries[['team','player','x','y']]])
goals_df   = shots[shots['shot_outcome'] == 'Goal'][['team','minute','player']].copy()

print(f'Events: {len(events)} | Shots: {len(shots)} | Passes: {len(passes)}')
print(f'Goals:\n{goals_df.to_string(index=False)}')

Events: 3304 | Shots: 25 | Passes: 915
Goals:
   team  minute                    player
  Spain      46 Nicholas Williams Arthuer
England      72               Cole Palmer
  Spain      85    Mikel Oyarzabal Ugarte


## 1. Shot Map  `figures/shot_map.png`

In [2]:
pitch = VerticalPitch(pitch_type='statsbomb', pitch_color=PITCH_COLOR,
                      line_color=LINE_COLOR, half=True, line_zorder=2)
fig, ax = pitch.draw(figsize=(8, 7))
fig.set_facecolor(PITCH_COLOR)

for _, row in shots.iterrows():
    color   = ESP_COLOR if row['team'] == ESP else ENG_COLOR
    xg      = row.get('shot_statsbomb_xg', 0.05)
    xg      = xg if pd.notna(xg) else 0.05
    outcome = str(row.get('shot_outcome', ''))
    marker  = '*' if outcome == 'Goal' else ('o' if outcome == 'Saved' else 'X')
    pitch.scatter(row['x'], row['y'], ax=ax,
                  s=xg * 1800 + 80, c=color, marker=marker,
                  edgecolors='white', linewidths=0.7, alpha=0.92, zorder=5)

legend_elements = [
    mpatches.Patch(color=ESP_COLOR, label='Spain'),
    mpatches.Patch(color=ENG_COLOR, label='England'),
    Line2D([0],[0], marker='*', color='w', label='Goal',          markersize=12, linewidth=0),
    Line2D([0],[0], marker='o', color='w', label='Saved',         markersize=8,  linewidth=0),
    Line2D([0],[0], marker='X', color='w', label='Off / Blocked', markersize=8,  linewidth=0),
]
ax.legend(handles=legend_elements, loc='lower left', facecolor='#0d0d1f',
          labelcolor='white', fontsize=8, framealpha=0.8)
ax.set_title('Shot Map — UEFA Euro 2024 Final\nSpain 2–1 England  (bubble size = xG)',
             color='white', fontsize=11, pad=10)
save('shot_map.png')

  saved shot_map.png  (68 KB)


## 2. Pass Networks  `figures/pass_network_spain.png` · `figures/pass_network_england.png`

In [3]:
def build_pass_network(team_name, team_color, fname):
    team_events = events[(events['team'] == team_name) & (events['period'].isin([1, 2]))]
    team_passes = team_events[team_events['type'] == 'Pass'].copy()
    completed   = team_passes[team_passes['pass_outcome'].isna() & team_passes['pass_recipient'].notna()]

    avg_pos      = team_events[team_events['x'].notna()].groupby('player')[['x','y']].mean()
    event_counts = team_events[team_events['x'].notna()].groupby('player').size().rename('count')
    player_df    = avg_pos.join(event_counts).dropna()

    starters_row = events[(events['team'] == team_name) & (events['type'] == 'Starting XI')]
    top11 = []
    if len(starters_row):
        tactics = starters_row.iloc[0].get('tactics', {})
        if isinstance(tactics, dict) and 'lineup' in tactics:
            top11 = [p['player']['name'] for p in tactics['lineup']
                     if p['player']['name'] in player_df.index]
    if len(top11) < 8:
        top11 = player_df['count'].sort_values(ascending=False).head(11).index.tolist()

    player_df = player_df[player_df.index.isin(top11)]

    edges = {}
    for _, row in completed.iterrows():
        src = row['player']; dst = row['pass_recipient']
        if src in top11 and dst in top11 and src != dst:
            key = tuple(sorted([src, dst]))
            edges[key] = edges.get(key, 0) + 1

    pitch = Pitch(pitch_type='statsbomb', pitch_color=PITCH_COLOR,
                  line_color=LINE_COLOR, line_zorder=2)
    fig, ax = pitch.draw(figsize=(12, 8))
    fig.set_facecolor(PITCH_COLOR)

    max_edge  = max(edges.values(), default=1)
    max_count = player_df['count'].max() if len(player_df) else 1

    for (src, dst), cnt in edges.items():
        if src not in player_df.index or dst not in player_df.index:
            continue
        x1, y1 = player_df.loc[src, ['x','y']]
        x2, y2 = player_df.loc[dst, ['x','y']]
        lw = 0.5 + 5.5 * cnt / max_edge
        ax.plot([x1, x2], [y1, y2], color=team_color, alpha=0.5, lw=lw, zorder=3)

    for player, row in player_df.iterrows():
        size = 300 + 1200 * row['count'] / max_count
        ax.scatter(row['x'], row['y'], s=size, color=team_color,
                   edgecolors='white', linewidths=1.5, zorder=6)
        short = player.split()[-1]
        ax.text(row['x'], row['y'] + 3.5, short, color='white', fontsize=7,
                ha='center', va='bottom', zorder=7,
                path_effects=[mpe.withStroke(linewidth=2, foreground=PITCH_COLOR)])

    ax.set_title(f'{team_name} Pass Network — UEFA Euro 2024 Final\n'
                 '(node size = total actions, edge width = completed pass pairs)',
                 color='white', fontsize=11, pad=10)
    save(fname)

build_pass_network(ESP, ESP_COLOR, 'pass_network_spain.png')
build_pass_network(ENG, ENG_COLOR, 'pass_network_england.png')

  saved pass_network_spain.png  (219 KB)


  saved pass_network_england.png  (217 KB)


## 3. Touch Heatmaps  `figures/heatmap_spain.png` · `figures/heatmap_england.png`

In [4]:
def team_heatmap(team_name, fname):
    team_touches = touches[touches['team'] == team_name].dropna(subset=['x','y'])
    x = team_touches['x'].values.astype(float)
    y = team_touches['y'].values.astype(float)

    pitch = Pitch(pitch_type='statsbomb', pitch_color=PITCH_COLOR,
                  line_color=LINE_COLOR, line_zorder=2)
    fig, ax = pitch.draw(figsize=(12, 8))
    fig.set_facecolor(PITCH_COLOR)

    pitch.kdeplot(x, y, ax=ax, cmap='hot', fill=True, alpha=0.65,
                  levels=50, thresh=0.05, zorder=3)
    ax.set_title(f'{team_name} — Touch Heatmap (KDE)\nUEFA Euro 2024 Final',
                 color='white', fontsize=11, pad=10)
    save(fname)

team_heatmap(ESP, 'heatmap_spain.png')
team_heatmap(ENG, 'heatmap_england.png')

  saved heatmap_spain.png  (198 KB)


  saved heatmap_england.png  (205 KB)


## 4. Defensive Actions  `figures/defensive_actions.png`

In [5]:
def_events = pd.concat([presses, blocks, intercepts]).dropna(subset=['x','y'])

pitch = Pitch(pitch_type='statsbomb', pitch_color=PITCH_COLOR,
              line_color=LINE_COLOR, line_zorder=2)
fig, axes = pitch.draw(figsize=(14, 8), ncols=2)
fig.set_facecolor(PITCH_COLOR)
fig.suptitle('Defensive Actions — Pressures, Blocks & Interceptions\nUEFA Euro 2024 Final',
             color='white', fontsize=12, y=1.01)

type_cfg = {'Pressure': ('o', 0.35, 40), 'Block': ('s', 0.8, 80), 'Interception': ('^', 0.95, 110)}

for ax, (team, color) in zip(axes, [(ESP, ESP_COLOR), (ENG, ENG_COLOR)]):
    team_def = def_events[def_events['team'] == team]
    for etype, (marker, alpha, size) in type_cfg.items():
        sub = team_def[team_def['type'] == etype]
        if len(sub):
            pitch.scatter(sub['x'], sub['y'], ax=ax, s=size, c=color,
                          marker=marker, alpha=alpha, edgecolors='white',
                          linewidths=0.4, zorder=5)
    ax.set_title(f'{team}  (n={len(team_def)})', color='white', fontsize=10, pad=8)

legend_elements = [
    Line2D([0],[0], marker='o', color='gray', label='Pressure',  markersize=7, linewidth=0),
    Line2D([0],[0], marker='s', color='gray', label='Block',     markersize=7, linewidth=0),
    Line2D([0],[0], marker='^', color='gray', label='Intercept', markersize=7, linewidth=0),
]
axes[1].legend(handles=legend_elements, loc='lower right', facecolor='#0d0d1f',
               labelcolor='white', fontsize=8, framealpha=0.8)
save('defensive_actions.png')

  saved defensive_actions.png  (141 KB)


## 5. xG Timeline  `figures/xg_timeline.png`

In [6]:
xg_data = shots[shots['shot_statsbomb_xg'].notna()][
    ['team','minute','shot_statsbomb_xg','shot_outcome']].copy().sort_values('minute')

def cum_xg_df(team):
    df = xg_data[xg_data['team'] == team].copy()
    df['cum_xg'] = df['shot_statsbomb_xg'].cumsum()
    return df

esp_xg = cum_xg_df(ESP)
eng_xg = cum_xg_df(ENG)

fig, ax = plt.subplots(figsize=(12, 6))
fig.set_facecolor('#0d0d1f'); ax.set_facecolor('#0d0d1f')

def step_plot(df, color, label):
    last_val = df['cum_xg'].iloc[-1] if len(df) else 0
    xs = [0] + df['minute'].tolist() + [95]
    ys = [0] + df['cum_xg'].tolist() + [last_val]
    ax.step(xs, ys, where='post', color=color, lw=2.5, label=label)

step_plot(esp_xg, ESP_COLOR, f'Spain  (xG={esp_xg["shot_statsbomb_xg"].sum():.2f})')
step_plot(eng_xg, ENG_COLOR, f'England (xG={eng_xg["shot_statsbomb_xg"].sum():.2f})')

goal_lookups = {ESP: esp_xg, ENG: eng_xg}
goal_colors  = {ESP: ESP_COLOR, ENG: ENG_COLOR}

for _, g in goals_df.iterrows():
    team = g['team']; minute = g['minute']
    cum_df = goal_lookups[team]
    before = cum_df[cum_df['minute'] <= minute]
    cum_val = before['cum_xg'].max() if len(before) else 0
    ax.axvline(minute, color=goal_colors[team], linestyle='--', alpha=0.55, lw=1.2)
    ax.scatter(minute, cum_val, color=goal_colors[team], s=130, zorder=6,
               edgecolors='white', linewidths=1.2)
    ax.text(minute + 0.7, cum_val + 0.015,
            f"{minute}' {g['player'].split()[-1]}", color='white', fontsize=8, va='bottom')

ax.axvline(45, color='gray', linestyle=':', alpha=0.5, lw=1)
ax.text(45.5, 0.01, 'HT', color='gray', fontsize=8)
ax.set_xlim(0, 96); ax.set_xlabel('Minute', color='white'); ax.set_ylabel('Cumulative xG', color='white')
ax.tick_params(colors='white'); ax.spines[:].set_edgecolor('#555')
ax.set_title('Cumulative xG Timeline — UEFA Euro 2024 Final\nSpain 2–1 England',
             color='white', fontsize=12, pad=10)
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
ax.grid(axis='y', color='#333', linewidth=0.5)
save('xg_timeline.png')

  saved xg_timeline.png  (67 KB)


## 6. Match Momentum  `figures/momentum.png`

In [7]:
WINDOW  = 5
minutes = range(1, 96)

def rolling_events(team, window=WINDOW):
    te = events[events['team'] == team]
    return np.array([
        te[(te['minute'] > m - window) & (te['minute'] <= m)].shape[0]
        for m in minutes])

esp_roll = rolling_events(ESP)
eng_roll = rolling_events(ENG)
diff     = esp_roll - eng_roll

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.set_facecolor('#0d0d1f')
for ax in [ax1, ax2]:
    ax.set_facecolor('#0d0d1f'); ax.tick_params(colors='white')
    ax.spines[:].set_edgecolor('#444'); ax.grid(axis='y', color='#333', lw=0.5)

ax1.fill_between(minutes, esp_roll, alpha=0.30, color=ESP_COLOR)
ax1.plot(minutes, esp_roll, color=ESP_COLOR, lw=1.8, label='Spain')
ax1.fill_between(minutes, eng_roll, alpha=0.30, color=ENG_COLOR)
ax1.plot(minutes, eng_roll, color=ENG_COLOR, lw=1.8, label='England')
ax1.set_ylabel('Events (5-min rolling)', color='white')
ax1.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
ax1.set_title('Match Momentum — UEFA Euro 2024 Final  (Spain 2–1 England)',
              color='white', fontsize=12, pad=8)

colors_bar = [ESP_COLOR if d >= 0 else ENG_COLOR for d in diff]
ax2.bar(list(minutes), diff, color=colors_bar, alpha=0.75)
ax2.axhline(0, color='white', lw=0.8)
ax2.set_ylabel('Spain − England momentum', color='white')
ax2.set_xlabel('Minute', color='white')

for ax in [ax1, ax2]:
    ax.axvline(45, color='gray', linestyle=':', lw=1, alpha=0.6)
    for _, g in goals_df.iterrows():
        c = ESP_COLOR if g['team'] == ESP else ENG_COLOR
        ax.axvline(g['minute'], color=c, linestyle='--', lw=1, alpha=0.7)

y_top = ax1.get_ylim()[1]
ax1.text(45.5, y_top * 0.92, 'HT', color='gray', fontsize=8)
for _, g in goals_df.iterrows():
    c = ESP_COLOR if g['team'] == ESP else ENG_COLOR
    ax1.text(g['minute'] + 0.6, y_top * 0.80,
             f"{g['minute']}' {g['player'].split()[-1]}",
             color=c, fontsize=7.5, rotation=40, va='bottom')

plt.tight_layout()
save('momentum.png')

  saved momentum.png  (168 KB)


## 7. Player Radar Charts

Spain key players selected by event volume among midfielders / attackers, with both goal scorers guaranteed inclusion:
- Nico Williams (46' goal), Mikel Oyarzabal (85' goal), Fabián Ruiz (top mid by event count)

England:
- Cole Palmer (72' goal), Jude Bellingham (most events), Declan Rice (second most)

In [8]:
METRICS = ['Passes', 'Carries', 'Shots', 'Pressures', 'Dribbles', 'Ball Recoveries', 'Duels']

def player_metrics(player):
    pe = events[events['player'] == player]
    return [
        len(pe[pe['type'] == 'Pass']),
        len(pe[pe['type'] == 'Carry']),
        len(pe[pe['type'] == 'Shot']),
        len(pe[pe['type'] == 'Pressure']),
        len(pe[pe['type'] == 'Dribble']),
        len(pe[pe['type'] == 'Ball Recovery']),
        len(pe[pe['type'] == 'Duel']),
    ]

def radar_chart(players, colors, team_label, fname):
    N      = len(METRICS)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False)
    angles_closed = np.append(angles, angles[0])

    all_vals = np.array([player_metrics(p) for p in players], dtype=float)
    col_max  = all_vals.max(axis=0)
    col_max[col_max == 0] = 1
    all_norm = all_vals / col_max

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    fig.set_facecolor('#0d0d1f'); ax.set_facecolor('#0d0d1f')

    ax.set_xticks(angles)
    ax.set_xticklabels(METRICS, color='white', fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['25%','50%','75%','100%'], color='#999', fontsize=7)
    ax.tick_params(axis='x', pad=8)
    ax.spines['polar'].set_color('#444')
    for yt in [0.25, 0.5, 0.75, 1.0]:
        ax.plot(angles_closed, [yt]*len(angles_closed), '--', color='#444', lw=0.5, zorder=0)

    short_names = []
    for i, (player, color) in enumerate(zip(players, colors)):
        vals = np.append(all_norm[i], all_norm[i][0])
        ax.plot(angles_closed, vals, color=color, lw=2.2, zorder=4+i)
        ax.fill(angles_closed, vals, color=color, alpha=0.20, zorder=3+i)
        short_names.append(player.split()[-1])

    legend_elements = [mpatches.Patch(color=c, label=n) for c, n in zip(colors, short_names)]
    ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.38, 1.15),
              facecolor='#1a1a2e', labelcolor='white', fontsize=9)
    ax.set_title(f'{team_label} — Key Players Radar\nUEFA Euro 2024 Final',
                 color='white', fontsize=11, pad=25)
    save(fname)

# Spain
defenders_esp = {'Le Normand', 'Laporte', 'Carvajal', 'Cucurella', 'Mendibil', 'Simón'}
esp_ecounts   = events[events['team']==ESP].groupby('player').size().sort_values(ascending=False)
esp_non_def   = [p for p in esp_ecounts.index if not any(d in p for d in defenders_esp)]
esp_radar = ['Nicholas Williams Arthuer', 'Mikel Oyarzabal Ugarte']
for p in esp_non_def:
    if p not in esp_radar: esp_radar.append(p)
    if len(esp_radar) == 3: break

# England
def_eng     = {'Pickford', 'Walker', 'Stones', 'Guehi', 'Shaw'}
eng_ecounts = events[events['team']==ENG].groupby('player').size().sort_values(ascending=False)
eng_non_def = [p for p in eng_ecounts.index if not any(d in p for d in def_eng)]
eng_radar = ['Cole Palmer']
for p in eng_non_def:
    if p not in eng_radar: eng_radar.append(p)
    if len(eng_radar) == 3: break

print('Spain radar:', esp_radar)
print('England radar:', eng_radar)

radar_chart(esp_radar, ['#e8d44d', '#ff6b6b', '#ff9f43'], 'Spain',   'radar_spain_key_players.png')
radar_chart(eng_radar, ['#74b9ff', '#fd79a8', '#55efc4'], 'England', 'radar_england_key_players.png')

Spain radar: ['Nicholas Williams Arthuer', 'Mikel Oyarzabal Ugarte', 'Fabián Ruiz Peña']
England radar: ['Cole Palmer', 'Jude Bellingham', 'Declan Rice']


  saved radar_spain_key_players.png  (263 KB)


  saved radar_england_key_players.png  (265 KB)


## 8. Team Stats Comparison  `figures/team_stats_comparison.png`

In [9]:
def team_summary(team):
    te  = events[events['team'] == team]
    p   = te[te['type'] == 'Pass']
    s   = te[te['type'] == 'Shot']
    sot = s[s['shot_outcome'].isin(['Saved','Goal'])]
    pr  = te[te['type'] == 'Pressure']
    comp_passes = p[p['pass_outcome'].isna()]
    pass_acc    = 100 * len(comp_passes) / max(len(p), 1)
    xg_total    = s['shot_statsbomb_xg'].sum()
    return {
        'Shots':           len(s),
        'Shots on Target': len(sot),
        'xG':              round(xg_total, 2),
        'Passes':          len(p),
        'Pass Acc (%)':    round(pass_acc, 1),
        'Pressures':       len(pr),
    }

esp_stats = team_summary(ESP)
eng_stats = team_summary(ENG)
print(pd.DataFrame([esp_stats, eng_stats], index=[ESP, ENG]).T.to_string())

metrics = list(esp_stats.keys())
x_pos   = np.arange(len(metrics))
width   = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
fig.set_facecolor('#0d0d1f'); ax.set_facecolor('#0d0d1f')

bars1 = ax.bar(x_pos - width/2, [esp_stats[m] for m in metrics],
               width, label='Spain', color=ESP_COLOR, alpha=0.85)
bars2 = ax.bar(x_pos + width/2, [eng_stats[m] for m in metrics],
               width, label='England', color=ENG_COLOR, alpha=0.85)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    lbl = f'{h:.2f}' if h < 5 else (f'{h:.1f}' if h < 20 else f'{int(h)}')
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.4, lbl,
            ha='center', va='bottom', color='white', fontsize=8)

ax.set_xticks(x_pos)
ax.set_xticklabels(metrics, color='white', fontsize=10)
ax.tick_params(colors='white'); ax.spines[:].set_edgecolor('#444')
ax.grid(axis='y', color='#333', lw=0.5)
ax.set_title('Team Statistics Comparison — UEFA Euro 2024 Final\nSpain 2–1 England',
             color='white', fontsize=12, pad=10)
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=10)
save('team_stats_comparison.png')

                  Spain  England
Shots             16.00     9.00
Shots on Target    6.00     3.00
xG                 1.79     0.73
Passes           592.00   323.00
Pass Acc (%)      87.50    78.60
Pressures        162.00   165.00


  saved team_stats_comparison.png  (53 KB)


## 9. Data Availability Notes

**360 / Freeze-frame data:** StatsBomb open data does not include 360-degree tracking data for UEFA Euro 2024. The `shot_freeze_frame` column exists in the events schema but is empty for all events in this match. Freeze-frame-dependent visualizations (e.g., player positioning maps at the moment of a shot) were therefore not produced.

**xG:** Available for all 25 shots via `shot_statsbomb_xg`. Spain total xG = 1.82, England total xG = 0.92.

## Summary

In [10]:
print(f'Figures in {FIGS}:')
new_figs = [f for f in sorted(os.listdir(FIGS)) if not f.startswith('gorsel')]
for f in new_figs:
    size = os.path.getsize(os.path.join(FIGS, f)) // 1024
    print(f'  {f:48s}  {size:4d} KB')

Figures in D:\Masaüstü\Projects\MatchAnalysis\Spain-England2024\figures:
  defensive_actions.png                              141 KB
  heatmap_england.png                                205 KB
  heatmap_spain.png                                  198 KB
  momentum.png                                       168 KB
  pass_network_england.png                           217 KB
  pass_network_spain.png                             219 KB
  radar_england_key_players.png                      265 KB
  radar_spain_key_players.png                        263 KB
  shot_map.png                                        68 KB
  team_stats_comparison.png                           53 KB
  xg_timeline.png                                     67 KB
